# Volti Project - Smart Meters in London Dataset Preparation Pipeline

Welcome! This notebook implements the memory-optimized data cleaning and feature engineering pipeline for the **Smart Meters in London** dataset. 

### Pipeline Steps:
1. **Load Metadata:** Load household classifications (`stdorToU`) and demographics (`Acorn_grouped`) and optimize their data types.
2. **Resample Weather:** Convert Dark Sky hourly weather records into half-hourly intervals using linear interpolation for numeric features and forward-fill for categories.
3. **Clean & Impute Consumption:** Handle missing values (`"Null"` strings) and cap extreme energy outliers ($> 10\text{ kWh}$ in a half-hour).
4. **Label Tariffs & Compute Cost:** Assign standard flat-rates ($14.228\text{ p/kWh}$) and Dynamic Time-of-Use rates (High: $67.20\text{p}$, Normal: $11.76\text{p}$, Low: $3.99\text{p}$) to calculate half-hourly energy costs.
5. **Feature Engineering:** Create time-series lag features (e.g., previous 30-min reading, 24-hours ago, and 1-week ago) and datetime variables for downstream modeling.
6. **Batch Export:** Save block data sequentially in optimized Parquet format to prevent Kaggle Kernel Out-Of-Memory (OOM) errors.

In [ ]:
import os
import gc
import time
import pandas as pd
import numpy as np

# =====================================================================
# PATH CONFIGURATION
# =====================================================================
KAGGLE_PATH = "/kaggle/input/smart-meters-in-london"
LOCAL_PATH = "c:/Users/ryntm/Desktop/bootcamp/yzta_bootcamp_304/archive"

if os.path.exists(KAGGLE_PATH):
    BASE_DIR = KAGGLE_PATH
    OUTPUT_DIR = "/kaggle/working/processed_data"
else:
    BASE_DIR = LOCAL_PATH
    OUTPUT_DIR = "c:/Users/ryntm/Desktop/bootcamp/yzta_bootcamp_304/processed_data"

HALF_HOURLY_DIR = os.path.join(BASE_DIR, "halfhourly_dataset", "halfhourly_dataset")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"[*] Base directory resolved to: {BASE_DIR}")
print(f"[*] Output directory set to: {OUTPUT_DIR}")

## 1. Load Metadata and Resample Weather

In [ ]:
def load_household_info():
    info_path = os.path.join(BASE_DIR, "informations_households.csv")
    print(f"[*] Loading household metadata from: {info_path}")
    
    hh_info = pd.read_csv(info_path, usecols=["LCLid", "stdorToU", "Acorn_grouped"])
    
    # Type downcasting for memory optimization
    hh_info["LCLid"] = hh_info["LCLid"].astype("category")
    hh_info["stdorToU"] = hh_info["stdorToU"].astype("category")
    hh_info["Acorn_grouped"] = hh_info["Acorn_grouped"].astype("category")
    return hh_info

def load_and_resample_weather():
    weather_path = os.path.join(BASE_DIR, "weather_hourly_darksky.csv")
    print(f"[*] Loading hourly weather parameters from: {weather_path}")
    
    weather = pd.read_csv(weather_path)
    weather["time"] = pd.to_datetime(weather["time"])
    weather = weather.set_index("time").sort_index()
    
    # Up-sampling hourly weather to half-hourly to match smart meter readings
    weather_resampled = weather.resample("30min").asfreq()
    
    # Interpolate numeric weather fields linearly
    numeric_cols = weather_resampled.select_dtypes(include=[np.number]).columns
    weather_resampled[numeric_cols] = weather_resampled[numeric_cols].interpolate(method="linear")
    
    # Forward-fill categoricals
    categorical_cols = weather_resampled.select_dtypes(exclude=[np.number]).columns
    weather_resampled[categorical_cols] = weather_resampled[categorical_cols].ffill()
    
    return weather_resampled.reset_index().rename(columns={"time": "tstp"})

## 2. Block Processing Pipeline Function

In [ ]:
def clean_and_process_block(block_file, hh_info, weather_df):
    block_path = os.path.join(HALF_HOURLY_DIR, block_file)
    if not os.path.exists(block_path):
        print(f"[!] Block file not found: {block_path}")
        return None
        
    df = pd.read_csv(block_path, dtype={"LCLid": "category", "energy(kWh/hh)": "string"})
    df["tstp"] = pd.to_datetime(df["tstp"])
    
    # Clean string 'Nulls' and strip whitespaces
    df["energy(kWh/hh)"] = pd.to_numeric(df["energy(kWh/hh)"].str.strip(), errors="coerce").astype("float32")
    
    # Forward fill missing slots per hane (household)
    df = df.sort_values(["LCLid", "tstp"])
    df["energy(kWh/hh)"] = df.groupby("LCLid", observed=False)["energy(kWh/hh)"].ffill().bfill()
    
    # Cap outliers at 10.0 kWh (unrealistic household load for 30 minutes)
    df.loc[df["energy(kWh/hh)"] > 10.0, "energy(kWh/hh)"] = 10.0
    
    # Merge household metadata
    df = df.merge(hh_info, on="LCLid", how="left")
    
    # Tariff Mapping
    df["price_pence"] = np.where(df["stdorToU"] == "Std", 14.228, 11.76).astype("float32")
    is_tou = df["stdorToU"] == "ToU"
    hours = df["tstp"].dt.hour
    
    # ToU Low window: 00:00 - 07:00 (3.99p)
    df.loc[is_tou & (hours >= 0) & (hours < 7), "price_pence"] = 3.99
    # ToU High window: 16:00 - 20:00 (67.20p)
    df.loc[is_tou & (hours >= 16) & (hours < 20), "price_pence"] = 67.20
    
    # Total cost calculation (£)
    df["cost_pounds"] = ((df["energy(kWh/hh)"] * df["price_pence"]) / 100.0).astype("float32")
    
    # Merge weather features
    df = df.merge(weather_df, on="tstp", how="left")
    
    # ML Time Series Lags (per household)
    df["lag_1"] = df.groupby("LCLid", observed=False)["energy(kWh/hh)"].shift(1).astype("float32")
    df["lag_2"] = df.groupby("LCLid", observed=False)["energy(kWh/hh)"].shift(2).astype("float32")
    df["lag_48"] = df.groupby("LCLid", observed=False)["energy(kWh/hh)"].shift(48).astype("float32")
    df["lag_336"] = df.groupby("LCLid", observed=False)["energy(kWh/hh)"].shift(336).astype("float32")
    
    # Datetime features
    df["hour"] = df["tstp"].dt.hour.astype("int8")
    df["day_of_week"] = df["tstp"].dt.dayofweek.astype("int8")
    df["month"] = df["tstp"].dt.month.astype("int8")
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype("int8")
    
    # Downcast objects to categories for optimal memory
    df["LCLid"] = df["LCLid"].astype("category")
    for col in ["precipType", "icon", "summary"]:
        if col in df.columns:
            df[col] = df[col].astype("category")
            
    return df

## 3. Run Pipeline for Selected Blocks

In [ ]:
hh_info = load_household_info()
weather_df = load_and_resample_weather()

# You can adjust this list to process more blocks (e.g. range(112))
blocks_to_process = [f"block_{i}.csv" for i in range(5)] 

for block_file in blocks_to_process:
    t_start = time.time()
    processed = clean_and_process_block(block_file, hh_info, weather_df)
    
    if processed is not None:
        out_name = block_file.replace(".csv", ".parquet")
        out_path = os.path.join(OUTPUT_DIR, out_name)
        processed.to_parquet(out_path, index=False, compression="snappy")
        print(f"[OK] Processed and exported {block_file} in {time.time() - t_start:.2f}s")
        
        del processed
        gc.collect()

## 4. Verification Check

In [ ]:
# Load a processed parquet file to verify correctness
test_path = os.path.join(OUTPUT_DIR, "block_0.parquet")
if os.path.exists(test_path):
    verify_df = pd.read_parquet(test_path)
    print("=== Output File Verification ===")
    print(f"File Shape: {verify_df.shape}")
    print("\nMemory Usage Summary:")
    verify_df.info(memory_usage="deep")
    print("\nHead Data Preview:")
    display(verify_df.head(10))
else:
    print(f"[!] Verification file not found at: {test_path}")